# BTC price (TODAY) Data Extraction, Load to BigQuery

## Data Extraction from Coingecko

In [ ]:
# --- IMPORT LIBRARIES ---
import requests 
import pandas as pd
import os
import pandas_gbq
from dotenv import load_dotenv
from datetime import datetime, date, time

In [ ]:
print("Starting INCREMENTAL Bitcoin data load pipeline...")

# --- LOAD SECRETS & CONFIG ---
load_dotenv()
API_KEY = os.getenv("GECKO_API_KEY") 
COIN_ID = "bitcoin"
#  parameters to load data to GBQ
GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID")
destination_table = "btc.raw_coingecko_bitcoin4"

url = f"https://api.coingecko.com/api/v3/coins/{COIN_ID}/market_chart"

# Validate secrets
if not API_KEY or not GCP_PROJECT_ID:
    raise ValueError("Error: GECKO_API_KEY or GCP_PROJECT_ID not set.")
print("✅ Secrets loaded successfully.")

Starting INCREMENTAL Bitcoin data load pipeline...
✅ Secrets loaded successfully.


In [27]:
# --- GET TODAY'S OPENING DATA ---

#Get today's date as a string (e.g., "2025-11-18")
today_midnight = datetime.combine(date.today(), time.min) # today at 00:00:00
today_str = today_midnight.strftime("%Y-%m-%d") 
print(f"Checking data for date: {today_str}")

# This endpoint is designed to get historical data for a specific date.
url = f"https://api.coingecko.com/api/v3/coins/{COIN_ID}/history"

# Set parameters for the API request - for today's opening price
params = {
    "date": today_str, 
    "localization": "false",
    "x_cg_demo_api_key": API_KEY
}

Checking data for date: 2025-11-18


In [ ]:
# --- EXTRACT (Get data from API) ---
try:
    print(f"Fetching final price for date: {today_midnight}...")
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    print("Data fetched successfully.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching data from CoinGecko API: {e}")
    exit(1) # Quit the script if we can't get data

Fetching final price for date: 2025-11-18 00:00:00...
Data fetched successfully.


In [23]:
data

{'id': 'bitcoin',
 'symbol': 'btc',
 'name': 'Bitcoin',
 'image': {'thumb': 'https://coin-images.coingecko.com/coins/images/1/thumb/bitcoin.png?1696501400',
  'small': 'https://coin-images.coingecko.com/coins/images/1/small/bitcoin.png?1696501400'},
 'market_data': {'current_price': {'aed': 338004.8744155231,
   'ars': 127654395.25815415,
   'aud': 141748.70612480005,
   'bch': 187.73526965719608,
   'bdt': 11251473.862616552,
   'bhd': 34702.53938822257,
   'bmd': 92036.7255045672,
   'bnb': 101.5054912213573,
   'brl': 490509.728576591,
   'btc': 1.0,
   'cad': 129334.88462546951,
   'chf': 73278.16825912833,
   'clp': 85219190.2888242,
   'cny': 654169.4338688124,
   'czk': 1918597.2117213053,
   'dkk': 593046.9240939743,
   'dot': 34640.47539270559,
   'eos': 391282.52627397917,
   'eth': 30.469213815429217,
   'eur': 79409.74694896811,
   'gbp': 69970.46038121969,
   'gel': 248959.34248985423,
   'hkd': 715516.5132538816,
   'huf': 30535101.087242603,
   'idr': 1541237801.6269321,

##  Data Transformation

In [24]:
# --- TRANSFORM (Parse the new JSON response) ---

# We parse it to get the single, final price.
try:
    price = data['market_data']['current_price']['usd']
except KeyError:
    print(f"Error: Could not find price in API response. JSON was: {data}")
    exit(1)

price

92036.7255045672

In [26]:
# Convert to DataFrame
df_to_load = pd.DataFrame({
    'date': [today_midnight],
    'price': [price]
})

print(f"Successfully parsed 1 row: {today_midnight}, ${price}")

Successfully parsed 1 row: 2025-11-18 00:00:00, $92036.7255045672


## Loading data into Warehouse (BigQuery)

In [ ]:
# --- 6. LOAD (Append data to BigQuery) ---

try:
    print(f"Appending 1 row to BigQuery table: {destination_table}...")
    pandas_gbq.to_gbq(
        df_to_load,
        destination_table=destination_table,
        project_id=GCP_PROJECT_ID,
        if_exists='append' 
    )
    print("✅ Pipeline finished successfully. 1 row appended to BigQuery.")
except Exception as e:
    print(f"Error loading data to BigQuery: {e}")
    exit(1)